# Text entity extraction

Which maintenance entities can deterministic lexical rules extract with auditable evidence?

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.data import write_json

pd.set_option("display.max_columns", 20)

from src.text_extraction import extract_records, LEXICONS

maintenance = pd.read_parquet(ROOT / "data/processed/maintenance.parquet")
out = ROOT / "results/text"
out.mkdir(parents=True, exist_ok=True)
duplicates = maintenance.drop(columns="row_id").duplicated()
duplicate_rows = maintenance.loc[duplicates, ["row_id", "WorkOrder"]]
maintenance = maintenance.loc[~duplicates].copy()
entities, records = extract_records(maintenance)
display(records.head(3))

,row_id,work_order,equipment_id,equipment_id_raw,order_type,maintenance_type,problem_text,action_text,combined_text,n_equipment_type,n_failure_mode,n_component,n_maintenance_action,n_asset_mention,identifier_concordance
0,0,20346569,M-067,M-067,,CM02,Motor M-02 earth fault trip. Single phase to e...,Isolate motor. Identify faulted phase using IR...,Motor M-02 earth fault trip. Single phase to e...,1,2,1,4,1,False
1,1,20381028,P-573,P-573,PM,PM01,Centrifugal Pump P-501 quarterly lubrication s...,Grease drive end and non-drive end bearings. M...,Centrifugal Pump P-501 quarterly lubrication s...,1,0,4,3,1,False
2,2,20232132,SC-045,SC-045,PM,PM01,Screw Conveyor SC-02 annual trough lienr inspe...,Inspect trough liner plates for wear. Replace ...,Screw Conveyor SC-02 annual trough lienr inspe...,1,0,3,4,1,False


## Evidence and coverage

Structured fields remain separate from text-derived entities. Longest spans are selected within each entity family; a component may also be an equipment type because those are different roles. A lexical failure mention does not establish an observed fault.

In [2]:
coverage = pd.DataFrame([
    {"entity_type": kind, "records_with_entity": int(records[f"n_{kind}"].gt(0).sum()),
     "coverage": float(records[f"n_{kind}"].gt(0).mean())}
    for kind in [*LEXICONS, "asset_mention"]
])
fields = {"WorkOrderDescription": "problem_text", "OperationDescription": "action_text"}
lookup = records.set_index("row_id")
assert all(lookup.loc[row.row_id, fields[row.source_field]][row.span_start:row.span_end] == row.evidence
           for row in entities.itertuples(index=False))
entities.to_csv(out / "entities.csv", index=False)
records.to_csv(out / "record_audit.csv", index=False)
coverage.to_csv(out / "extraction_coverage.csv", index=False)
duplicate_rows.to_csv(out / "exact_duplicates.csv", index=False)
display(coverage)
display(entities.head(15))

,entity_type,records_with_entity,coverage
0,equipment_type,27207,0.556950
1,failure_mode,15347,0.314166
2,component,38449,0.787083
3,maintenance_action,47709,0.976643
4,asset_mention,47247,0.967185


,row_id,work_order,entity_type,entity_label,source_field,evidence,span_start,span_end
0,0,20346569,equipment_type,motor,WorkOrderDescription,Motor,0,5
1,0,20346569,failure_mode,earth_fault,WorkOrderDescription,earth fault,11,22
2,0,20346569,failure_mode,trip,WorkOrderDescription,trip,23,27
3,0,20346569,component,motor,WorkOrderDescription,Motor,0,5
4,0,20346569,component,motor,OperationDescription,motor,8,13
5,0,20346569,component,motor,OperationDescription,motor,71,76
6,0,20346569,maintenance_action,isolate,OperationDescription,Isolate,0,7
7,0,20346569,maintenance_action,test,OperationDescription,testing,57,64
8,0,20346569,maintenance_action,install,OperationDescription,Install,102,109
9,0,20346569,maintenance_action,inspect,OperationDescription,inspected,110,119


## Review sample

This is a sample prepared for manual annotation, not an already validated extraction benchmark. Blank review fields are intentional.

In [3]:
review = records.sample(n=min(100, len(records)), random_state=config.SEED).sort_values("row_id")
labels = entities.groupby(["row_id", "entity_type"])["entity_label"].agg(lambda x: "|".join(sorted(set(x)))).unstack(fill_value="").reset_index()
review = review.merge(labels, on="row_id", how="left", validate="one_to_one").fillna("")
review["review_status"] = "pending"
review["review_notes"] = ""
review.to_csv(out / "manual_review.csv", index=False)
write_json(out / "summary.json", {
    "input_records": int(len(maintenance) + len(duplicate_rows)), "records": len(records),
    "exact_duplicates_removed": len(duplicate_rows), "entities": len(entities),
    "unique_valid_assets": int(records.loc[records["equipment_id"].ne(""), "equipment_id"].nunique()),
    "identifier_concordance": float(records["identifier_concordance"].mean()),
    "manual_review": "100 sampled records prepared; annotation pending"
})
display(review.head(5))

,row_id,work_order,equipment_id,equipment_id_raw,order_type,maintenance_type,problem_text,action_text,combined_text,n_equipment_type,...,n_maintenance_action,n_asset_mention,identifier_concordance,asset_mention,component,equipment_type,failure_mode,maintenance_action,review_status,review_notes
0,147,20408866,CF-078,CF-078,PM,PM01,"Centrifuge CF-01 six-monthly basket, bearing a...",Inspect basket condition. Measure bearing clea...,"Centrifuge CF-01 six-monthly basket, bearing a...",0,...,3,1,False,CF-01,bearing|seal,,,inspect|lubricate|test,pending,
1,708,20699470,EX-134,EX-134,PM,PM02,Extruder EX-02 six-monthly gearbox oil change ...,Drain gearbox. Replace oil. Inspect thrust bea...,Extruder EX-02 six-monthly gearbox oil change ...,1,...,4,1,False,EX-02,bearing|coupling|gearbox,gearbox,,align|inspect|replace|test,pending,
2,879,20109416,SEP-022,SEP-022,CM,CM01,Separator SEP-02 liquid dump valve failing to ...,Isolate dump valve. Remove actuator. Inspect v...,Separator SEP-02 liquid dump valve failing to ...,1,...,4,1,False,SEP-02,valve,valve,,inspect|isolate|replace|test,pending,
3,1277,20750515,BLR-143,BLR-143,CM,CM02,Boiler BLR-01 feedwater pump failure. Loss of ...,Start standby feedwater pump. Investigate duty...,Boiler BLR-01 feedwater pump failure. Loss of ...,1,...,2,1,False,BLR-01,bearing|pump|seal,pump,,replace|test,pending,
4,1415,20268541,TRB-051,TRB-051,CM,CM03,Steam Turbine TRB-01 lube oil low pressure ala...,Clean oil pump suction strainer. Check oil pum...,Steam Turbine TRB-01 lube oil low pressure ala...,1,...,4,1,False,TRB-01,bearing|pump|valve,pump,blockage|pressure_loss,clean|inspect|lubricate|test,pending,
